<a href="https://colab.research.google.com/github/Engr-Muhammad-Anees/Dubbing-Podcast-ML/blob/main/db_podcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**install packages:**

In [ ]:
!pip install moviepy pydub webrtcvad openai gTTS groq
!apt-get install ffmpeg -y

**import libraries:**

In [ ]:
from groq import Groq
import os, json
import numpy as np
import librosa
from moviepy.editor import VideoFileClip
from pydub import AudioSegment
from gtts import gTTS
import subprocess, os

**audio_extract:**

In [ ]:
input_video = "/content/input_video.mp4"
video = VideoFileClip(input_video)
video.audio.write_audiofile("audio.wav")


**GROQ API KEY:**

In [ ]:
os.environ["GROQ_API_KEY"] = "YOUR API KEY"
client = Groq(api_key=os.environ["GROQ_API_KEY"])

**Transcribe with groq api:**

In [ ]:
with open("audio.wav", "rb") as f:
    transcription = client.audio.transcriptions.create(
        model="whisper-large-v3",
        file=f,
        response_format="verbose_json"
    )

data = transcription.model_dump()
with open("transcription_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

**list transcribe segment:**

In [ ]:
segments = data["segments"]
total_duration = data["duration"]
print(f"Transcribed {len(segments)} segments, total {total_duration:.2f}s")

**normalize and change it in momo:**

In [ ]:
y, sr = librosa.load("audio.wav", sr=16000)
frame_length = int(0.5 * sr)
hop_length = int(0.25 * sr)
rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]

rms_norm = (rms - rms.min()) / (rms.max() - rms.min() + 1e-8)
threshold = 0.4
changes = np.where(np.diff(rms_norm > threshold))[0]

speaker_map = []
current_speaker = "A"
last_change_frame = 0
for ch in changes:
    start = last_change_frame * hop_length / sr
    end = ch * hop_length / sr
    speaker_map.append((start, end, current_speaker))
    current_speaker = "B" if current_speaker == "A" else "A"
    last_change_frame = ch
speaker_map.append((last_change_frame * hop_length / sr, len(y)/sr, current_speaker))


In [ ]:
speaker_map

**merge all segments into one:**

In [ ]:
merged_segments = []
for seg in segments:
    start, end, text = seg["start"], seg["end"], seg["text"].strip()
    speaker = "A"
    for (s_start, s_end, spk) in speaker_map:
        if s_start < end and s_end > start:
            speaker = spk
            break
    merged_segments.append({"start": start, "end": end, "text": text, "speaker": speaker})

In [ ]:
merged_segments

In [ ]:
os.makedirs("tts_segments", exist_ok=True)
speaker_voices = {"A": "en", "B": "en"}
def generate_tts(text, path, lang='en'):
    tts = gTTS(text=text, lang=lang)
    tts.save(path)

def match_duration(tts_path, target_duration):
    stretched_path = tts_path.replace(".wav", "_fit.wav")
    dur = AudioSegment.from_file(tts_path).duration_seconds
    if dur <= 0: dur = target_duration
    rate = max(0.5, min(target_duration / dur, 2.0))
    subprocess.run([
        "ffmpeg", "-y", "-i", tts_path,
        "-filter:a", f"atempo={rate:.3f}",
        stretched_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return stretched_path

aligned = AudioSegment.silent(duration=int(total_duration * 1000))

**dubbed audio:**

In [ ]:
for i, seg in enumerate(merged_segments):
    if not seg["text"].strip():
        continue
    start_ms = int(seg["start"] * 1000)
    dur = seg["end"] - seg["start"]

    tts_path = f"tts_segments/seg_{i}_{seg['speaker']}.wav"
    generate_tts(seg["text"], tts_path, lang=speaker_voices[seg["speaker"]])
    tts_fit = match_duration(tts_path, dur)
    audio = AudioSegment.from_file(tts_fit)
    aligned = aligned.overlay(audio, position=start_ms)

aligned.export("dubbed_audio.wav", format="wav")

<_io.BufferedRandom name='dubbed_audio.wav'>

**dubbed video:**

In [ ]:
!ffmpeg -y -i "/content/input_video.mp4" -i "dubbed_audio.wav" -c:v copy -map 0:v:0 -map 1:a:0 -shortest "output_dubbed.mp4"
print("Dubbing complete! File saved as output_dubbed.mp4")